# Project 1 
### Kwatcho Mahinanda

## Project Goal 
The goal of this project is to analyze flight data collected from the Delta Airlines website to understand cost and duration patterns for a specific round trip flight.

### Overview 
This analysis uses data scraped from the Delta Airlines website ([https://www.delta.com/](https://www.delta.com/)) on October 20, 2025.

### Data Collected
The dataset includes the following details for various flight options:
* Departure Location
* Arrival Location
* Trip Direction (Round Trip)
* Departure Date
* Return Date
* Cost of Flight ($)
* Duration of Flight

### Specific Search Criteria
The data collection focused on a single search query:
* **Trip Type:** Round Trip
* **Origin:** Hartsfield-Jackson Atlanta International Airport (ATL), Atlanta, GA
* **Destination:** Raleigh-Durham International Airport (RDU), Raleigh, NC
* **Departure Date:** December 21, 2025
* **Return Date:** December 28, 2025


# Methodology

This project involves scraping flight data from the Delta Airlines website using browser automation and analyzing the results. The steps are as follows:

## 1. Setup and Website Interaction
* **Import Packages:** Import necessary Python libraries, primarily `selenium` for browser automation and `pandas` for data manipulation.
* **Initialize Driver:** Start a Selenium WebDriver instance to control a web browser.
* **Handle Consent:** Navigate to the Delta website and programmatically click the 'I understand' button to accept cookie/data usage policies.
* **Define Input Functions:**
    * Create a function to input the origin (ATL) and destination (RDU) airport codes into the appropriate fields.
    * Create a function to select the trip direction ("Round Trip") from the relevant dropdown menu.
    * Create a function to interact with the calendar widget to select the specific departure (December 21, 2025) and return (December 28, 2025) dates.
* **Execute Search:** Trigger the flight search button after setting all parameters.

## 2. Data Scraping and Structuring
* **Extract Data:** Once the search results page loads, scrape the following information for each available flight option:
    * Departure Location (Should be ATL)
    * Arrival Location (Should be RDU)
    * Trip Direction (Should be Round Trip)
    * Departure Date (Should be Dec 21, 2025)
    * Return Date (Should be Dec 28, 2025)
    * Cost of the Flight
    * Duration of the Flight
* **Create Dataset:** Store the extracted data into a structured format with columns corresponding to the variables listed above.

## 3. Data Validation 
* **Verify Flight Count:** Manually check the number of flight results displayed on the Delta website for the specified search criteria. Compare this count to the total number of rows (flights) captured in the pandas DataFrame to ensure all available flights were scraped successfully.

## 4. Data Analysis 
* **Descriptive Statistics:** Analyze the collected flight data, focusing on the cost. I will calculate and present the following summary statistics for the 'Cost of Flight' column:
    * Mean (Average cost)
    * Variance (Spread of costs around the mean)
    * Minimum (Lowest cost flight found)
    * Maximum (Highest cost flight found)

In [1]:
#------------------- IMPORT PACKAGES FOR DATA PROCESSING ----------------------#

import os, re
import traceback

# Manage datasets
import pandas as pd

# Work with time data
import time 

# Conduct HTTP requests
import requests

# Construct tree structure of HTML data
import html5lib

# Parse HTML data obtained from scraping
from bs4 import BeautifulSoup

# Import webdriver for chrome
from webdriver_manager.chrome import ChromeDriverManager


from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager


# Automate navigating within browser (SELENIUM)
#------ Key: Manage keys
#------ Select: Obtain features from website
#------ WebDriverWait: Add wait times implicitly
#------ By: Use common information locator strategies
#------ EC and Options: Browser configuration
#------ remote.command: Check whether browser is active

from selenium import webdriver #to automate the navigating within the browser
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.keys    import Keys
from selenium.webdriver.support.ui     import Select
from selenium.webdriver.support.ui     import WebDriverWait 
from selenium.webdriver.common.by      import By
from selenium.webdriver.support        import expected_conditions as EC
from selenium.webdriver.chrome.options import Options #to use properties of the chrome webbrowser
from selenium.webdriver.remote.command import Command # Use to check whether the web driver is active

In [2]:
# Initialize Driver and Starting URL

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service = Service(ChromeDriverManager().install()))
options = webdriver.ChromeOptions()
starting_url = 'https://www.delta.com/flightsearch/book-a-flight'
driver.get(starting_url)

In [3]:
# Handle Cookie Consent Banner
# Wait a couple of seconds for the page (and the cookie banner) to load fully
time.sleep(2)

try:
    # Find the "I understand" button by its ID
    accept_cookies_button = driver.find_element(By.ID, 'onetrust-accept-btn-handler')

    # Click the button
    accept_cookies_button.click()
    print("Clicked the 'I understand' button for cookies.")

    # Add a small pause to let the banner disappear
    time.sleep(1)

except Exception as e:
    print(f"Could not find or click the cookie button: {e}")
    # Consider if you want the script to stop or continue if the button isn't found


Clicked the 'I understand' button for cookies.


In [4]:
# Function to Set Airport
def set_airport(driver, airport_button_id, airport_code):
    """Clicks the airport button, types the code, and selects from suggestions."""
    try:
        # 1. Click the main airport button (e.g., "From" or "To")
        airport_element = driver.find_element(By.ID, airport_button_id)
        airport_element.click()
        print(f"Clicked the '{airport_button_id}' element.")
        time.sleep(1) # Wait for search box

        # 2. Find and type into the search input
        search_input = driver.find_element(By.ID, "search_input")
        search_input.clear()
        search_input.send_keys(airport_code)
        print(f"Typed '{airport_code}' into the search input.")
        time.sleep(1) # Wait for suggestions

        # 3. Find and click the correct suggestion using XPath
        suggestion_xpath = f"//a[@class='airportLookup-list'][contains(., '{airport_code}')]"
        wait = WebDriverWait(driver, 5) # Wait up to 5 seconds
        suggestion = wait.until(EC.element_to_be_clickable((By.XPATH, suggestion_xpath)))
        suggestion.click()
        print(f"Selected '{airport_code}' from the suggestion list.")
        time.sleep(1) # Wait for selection to register
        return True # Indicate success

    except Exception as e:
        print(f"Error setting airport '{airport_code}' using button '{airport_button_id}': {e}")
        return False # Indicate failure

# Call the function for the Origin 
set_airport(driver, "fromAirportName", "ATL")

# Next: Find the ID for the Destination button and call it again 
set_airport(driver, "toAirportName", "RDU") 

Clicked the 'fromAirportName' element.
Typed 'ATL' into the search input.
Selected 'ATL' from the suggestion list.
Clicked the 'toAirportName' element.
Typed 'RDU' into the search input.
Selected 'RDU' from the suggestion list.


True

In [5]:
# Function to Select Trip Type
def select_trip_type(driver, trip_type_text="Round Trip", option_id="ui-list-selectTripType0"):
    try:
        # 1. Find and click the main wrapper element to open the dropdown
        # XPath targeting the span that contains the span with id='selectTripType-val'
        wrapper_xpath = "//span[contains(@class, 'select-ui-wrapper')][.//span[@id='selectTripType-val']]"

        wait = WebDriverWait(driver, 10)
        trip_type_wrapper = wait.until(EC.element_to_be_clickable((By.XPATH, wrapper_xpath)))
        trip_type_wrapper.click()
        print("Clicked the trip type wrapper to open dropdown.")
        time.sleep(1) # Wait for dropdown options to appear

        # 2. Find and click the specific trip type option using its unique ID
        trip_type_option = wait.until(EC.element_to_be_clickable((By.ID, option_id)))
        trip_type_option.click()
        print(f"Selected '{trip_type_text}' from dropdown.")
        time.sleep(1) # Wait for selection
        print("Successfully selected trip type")
        return True

    except Exception as e:
        print(f"Error selecting trip type '{trip_type_text}': {e}")
        traceback.print_exc()
        return False

# Call the function
# Assumes the ID for "Round Trip" is indeed "ui-list-selectTripType0"
select_trip_type(driver, trip_type_text="Round Trip", option_id="ui-list-selectTripType0")

Clicked the trip type wrapper to open dropdown.
Selected 'Round Trip' from dropdown.
Successfully selected trip type


True

In [6]:
# Function to Select Dates with Month Navigation

def select_calendar_dates_with_nav(driver, opener_id, target_month, target_year, departure_aria_label, return_aria_label):
    wait = WebDriverWait(driver, 15) # Wait up to 15 seconds
    try:
        # 1. Click to Open Calendar
        calendar_opener = wait.until(EC.element_to_be_clickable((By.ID, opener_id)))
        calendar_opener.click()
        print(f"Clicked element with ID '{opener_id}' to open calendar.")
        time.sleep(1) # Allow slight render time

        # 2. Navigate Months until Target Month/Year is Visible
        max_clicks = 12 # Safety break to prevent infinite loop
        clicks = 0
        while clicks < max_clicks:
            try:
                # Find current month and year displayed (in the first pane)
                current_month_element = driver.find_element(By.CLASS_NAME, "dl-datepicker-month-0")
                current_year_element = driver.find_element(By.CLASS_NAME, "dl-datepicker-year-0")
                current_month = current_month_element.text
                current_year = current_year_element.text
                print(f"  Current calendar view: {current_month} {current_year}")

                # Check if it's the target month/year
                if current_month.lower() == target_month.lower() and current_year == str(target_year):
                    print(f"Found target month/year: {target_month} {target_year}")
                    break # Exit the loop

                # If not, click the 'Next' button
                print("  Target not visible, clicking 'Next' month...")
                next_button_class = "dl-datepicker-1" # Class for the '>' arrow
                next_button = wait.until(EC.element_to_be_clickable((By.CLASS_NAME, next_button_class)))
                next_button.click()
                clicks += 1
                time.sleep(0.7) # Wait for month to change

            except Exception as nav_e:
                print(f"Error during month navigation: {nav_e}")
                # Try waiting a bit longer in case content is slow to update
                time.sleep(2)
                # Re-check elements or break if persistent error
                if clicks > 0: # Avoid infinite loop if elements vanish on first try
                    print("Retrying month/year element find...")
                else:
                    raise nav_e # Raise error if it fails immediately

            if clicks >= max_clicks:
                print(f"⚠️ Reached max clicks ({max_clicks}) without finding target month/year.")
                return False # Exit function if target not found


        # 3. Click Departure Date
        departure_selector = f"a[aria-label='{departure_aria_label}']"
        departure_date_element = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, departure_selector)))
        departure_date_element.click()
        print(f"Clicked departure date: {departure_aria_label}")
        time.sleep(0.5)

        # 4. Click Return Date
        return_selector = f"a[aria-label='{return_aria_label}']"
        return_date_element = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, return_selector)))
        return_date_element.click()
        print(f"Clicked return date: {return_aria_label}")
        time.sleep(1)

        # 5. Click the 'Done' button
        done_button_xpath = "//button[@class='donebutton'][contains(text(), 'done')]"
        done_button = wait.until(EC.element_to_be_clickable((By.XPATH, done_button_xpath)))
        done_button.click()
        print("Clicked the calendar 'Done' button.")
        time.sleep(1.5)

        print("Successfully selected dates and closed calendar")
        return True

    except Exception as e:
        print(f"Error in date selection process: {e}")
        # Print the full traceback to help pinpoint the error's origin during debugging
        traceback.print_exc()
        return False

# Define Target Month/Year and Labels
calendar_opener_id = "input_departureDate_1"
target_month_name = "December"
target_year_num = 2025
departure_label = "21 December 2025, Sunday"
return_label = "28 December 2025, Sunday"

# Call the function
select_calendar_dates_with_nav(driver, calendar_opener_id, target_month_name, target_year_num, departure_label, return_label)

Clicked element with ID 'input_departureDate_1' to open calendar.
  Current calendar view: October 2025
  Target not visible, clicking 'Next' month...
  Current calendar view: November 2025
  Target not visible, clicking 'Next' month...
  Current calendar view: December 2025
Found target month/year: December 2025
Clicked departure date: 21 December 2025, Sunday
Clicked return date: 28 December 2025, Sunday
Clicked the calendar 'Done' button.
Successfully selected dates and closed calendar


True

In [8]:
# Click the Main Search Button and Wait for Results Page to Load

try:
    search_button_id = "btnSubmit"
    # Use a longer wait here as the search results page might take time
    wait = WebDriverWait(driver, 20) # Wait up to 20 seconds

    search_button = wait.until(EC.element_to_be_clickable((By.ID, search_button_id)))
    search_button.click()
    print("Clicked the main search button.")

    # Wait for the "Outbound" label to appear on the results page so we can be sure it's loaded
    # Here is XPath to find the div with the specific class containing the text "Outbound"
    #outbound_label_xpath = "//div[@class='mach-flight-context-info__wrapper__info--label'][contains(text(), 'Outbound')]"
    #wait.until(EC.presence_of_element_located((By.XPATH, outbound_label_xpath)))
    #print("Results page loaded (found 'Outbound' label).")
    # Short pause after element found, before scraping
    time.sleep(10)

except Exception as e:
    print(f"Error clicking search button or waiting for results: {e}")
    # Print the full traceback
    traceback.print_exc()

Clicked the main search button.
